# 无分类器引导 (Classifier-Free Guidance, CFG)

## 1. 背景：为什么需要“无分类器”？
在 CFG 提出之前，为了让扩散模型生成更符合条件（如特定类别、文本描述）且质量更高的图像，研究者们使用的是**分类器引导 (Classifier Guidance)**。
分类器引导的做法是：不仅仅训练一个条件扩散模型，还要额外配对训练一个**图片分类器**。在生成图像的反向扩散过程中，利用该分类器对当前带噪图像进行分类，并将分类概率的**梯度**加入到扩散模型的采样步骤中，从而“把图像往我们想要的类别上拽”。

**分类器引导面临的痛点：**
1. **必须费力训练额外的分类器**：由于扩散模型的生成路线是由纯噪声向清晰图像演化，所以分类器不能用现成的（例如公开的 ResNet），必须在“带有不同程度噪声”的序列图像上从头训练一套专属网络。
2. **类似对抗攻击陷阱**：引入分类器的梯度作引导，本质上是在执行类似对抗攻击 (Adversarial Attack) 的操作。模型为了迎合分类器，可能会走捷径，产生出可以骗过分类器的大量“高频噪声或马赛克图”，这会导致分类器评估指标虚高，但视觉上人眼看起来极不自然。
3. **训练推理管线被割裂**：生成器和判别器分家，增加了部署和调参的难度。

---

## 2. CFG 的核心思想
**用一句话概括 CFG**：我们不再需要外部的分类器，而是让同一个扩散模型，在训练时既学会**“有条件生成（听指令）”**，又学会**“无条件生成（自由发挥）”**。在推理采样时，我们分别算出这两种预期的结果，并**人为放大它们之间的特征差异**，从而极大地强化条件指令的引导力度。

**直觉比喻**：
例如我们打字要求扩散模型画一只“红色的猫”。
- **无条件生成**：模型随心所欲，可能画出了一个迷糊的普通动物（不知道要红，也不知道要猫）。
- **有条件生成**：模型看到了要求，画出了带有一定猫特征和少量红色的粗略边缘。
- **CFG 放大差异**：CFG 的数学机制是将“有条件的画”减去“无条件的画”，差值部分就留下了**“区别于一般事物的、专属于红色猫独有的特征偏置”**。然后，我们将这个独有特征乘以一个常数倍率拉满，再加回到基础图像上。如此一来，画出来的猫特征会被放大到极致（红得极度鲜艳，猫特征更清晰）。

---

## 3. 数学原理与终极公式推导

扩散发明的数学本质，是用神经网络 $\epsilon_\theta(z_t, t)$ 去拟合带噪数据分布对数概率密度的梯度（也就是得分函数 Score Function，二者互为等价关系，仅差一个时间常数）：
$$ \epsilon_\theta(z_t, c) \approx -\sigma_t \nabla_{z_t} \log p(z_t|c) $$

根据**贝叶斯定理**，我们将其拆解：
$$ p(c|z_t) = \frac{p(z_t|c) p(c)}{p(z_t)} $$
对等式两边取对数，并对变量 $z_t$ 求梯度。由于先验概率分布 $p(c)$ 与 $z_t$ 无关，其关于 $z_t$ 的梯度为 0，于是得到：
$$ \nabla_{z_t} \log p(c|z_t) = \nabla_{z_t} \log p(z_t|c) - \nabla_{z_t} \log p(z_t) $$
> **这里展现出了令人惊叹的等价置换**：等式左边 $\nabla_{z_t} \log p(c|z_t)$ 恰巧就是我们原本在上一代技术里需要额外训练的“分类器的引导梯度”。而等式右边证明，这其实严格等价于**“包含条件的得分” 减去 “无条件的得分”**！

现在，我们用神经网络代替物理概率公式，将右边的项表现为预测的噪声差值：
$$ \nabla_{z_t} \log p(c|z_t) = \left( -\frac{1}{\sigma_t} \epsilon_\theta(z_t, c) \right) - \left( -\frac{1}{\sigma_t} \epsilon_\theta(z_t) \right) = - \frac{1}{\sigma_t} \big( \epsilon_\theta(z_t, c) - \epsilon_\theta(z_t) \big) $$

接下来，如同传统的分类器引导一样，我们将通过上面推导出的“隐式分类器梯度”放大 $w$ 倍后，补偿到到无条件得分中：
$$ \tilde{\epsilon}_\theta(z_t, c) = \epsilon_\theta(z_t) + (1 + w) \big( \epsilon_\theta(z_t, c) - \epsilon_\theta(z_t) \big) $$

整理合并同类项，我们就得到了 CFG 论文中惊艳、强大且极其优雅的**核心采样公式**：
$$ \boxed{ \tilde{\epsilon}_\theta(z_t, c) = (1 + w)\epsilon_\theta(z_t, c) - w \epsilon_\theta(z_t) } $$

- **当 $w = 0$ 时**：公式退化为纯 $\epsilon_\theta(z_t, c)$，这意味着只是普通基于条件的扩散生成。
- **当 $w > 0$ 时**：模型倒扣了无条件特征（$-w \epsilon_\theta(z_t)$），放大了由于“条件 $c$”带来的特征偏移量。在现在的开源绘图框架（例如 Stable Diffusion 和 Midjourney）中，这正是最核心的调参参数 **Guidance Scale (引导尺度)**，取值在 7 到 10 时往往能够获得色彩最鲜艳、结构最突出的震撼大图。

---

## 4. 如何进行训练？(Training Strategy)

公式显示在推理时刻（Sampling）要同时用到“条件噪声” $\epsilon_\theta(z_t, c)$ 和“无条件噪声” $\epsilon_\theta(z_t)$，难道我们要因此付出代价训练两个不同的 U-Net 网络吗？
当然不必！这里体现了 CFG 工程设计的极度美妙——只需加入一种名叫 **条件丢失（Condition Dropout）** 的训练机制。

开发者对原版的扩散模型（如 DDPM模型代码）只做**一行逻辑上的修改**：
1. **设定随机丢失概率**：设置一个名为 $p_{\text{uncond}}$ 的超参丢弃概率（作者经过实验证明设为 $10\%$ 到 $20\%$ 时模型性比表现最强）。
2. **随机替换标签 (Drop Conditioning)**：在提取 Batch 数据喂给神经网络时，以概率 $p_{\text{uncond}}$ 前置触发拦截，将输入的标签信息（如 Class 类别或是 Text Embedding）被生硬地设定为一个固定的、特殊的**空标签 $\emptyset$** (Null Token 或 空字符串向量)。

通过这一个操作：神经网络若看到具体标签 $c$，就利用反向传播学习建立语义联动（条件拟合）；若是被蒙上眼睛看到了 $\emptyset$，就学习如何仅从构图噪点还原世界（无条件拟合）。全过程一套权重全包，不增加任何多余的参数规模！

---

## 5. CFG 的优势与代偿代价

### CFG 带来的巨大优势：
1. **破除了体系孤岛**：彻底埋葬了额外的图像分类网络（Classifier），只用纯粹的生成模型本身。这也从侧面反映了一个哲学：完美的“生成器”内部自带了一个最懂自己的“判别器”。
2. **画质与保真度的飞跃式提升**：因为它不是强硬地“顺着外部梯度的方向盲干”，所以躲开了让模型生成生硬马赛克噪点的“对抗攻击”效应。加上 CFG 引导时，图像表现得非常平滑，对比度和清晰度陡增！
3. **终极 Trade-off 杠杆**：只留给用户一个可控权重 $w$，供使用者直接在**“图像生成质量/保真度 (Fidelity)”**和**“样本画面的变幻多样性 (Diversity)”**之间丝滑平调。

### 唯一的沉重代价：
- **推理耗时翻一番 (Sampling Speed Penalty)**：这就是 CFG 被誉为拿时间换效果的终极手段。回到公式 $\tilde{\epsilon}_\theta = (1 + w)\epsilon_\theta(z_t, c) - w \epsilon_\theta(z_t)$ ——这意味着不管是在 DDIM 还是 DPM-Solver 中，计算任何一步的梯度变化时，**都必须调用两次大模型网络做 Forward 前向推理推演**（分别喂入文本 $c$ 和空文本 $\emptyset$ ）。这也就是为什么如今的各种 WebUI 里打开高CFG值生图速度往往需要更多计算力的直接原理。

## 6. 前沿拓展：CFG 在 Flow Matching (流匹配) 中的等价推导

虽然 CFG 最初是被提出用于扩散模型（预测噪声 $\epsilon$），但它的核心思想——**“寻找并放大条件分布与无条件分布的增量差异”**——完全可以无缝迁移到基于常微分方程 (ODE) 的 **Flow Matching (流匹配)** 框架中。以**自动驾驶轨迹生成 (Trajectory Generation)** 为例，CFG 能极其优雅地解决复杂交互场景下的轨迹规划问题。

### 6.1 概率重塑：一切的起点
在 Flow Matching 中，如果我们把生成的物理量从“图像像素 $x$”替换为“驾驶轨迹 $\tau_1$”，并引入场景条件 $C$。研究者定义了一个增强的条件概率分布：
$$ \tilde{q}(\tau_1 | C) \propto q(\tau_1)^{1-\omega} q(\tau_1 | C)^\omega $$
其中 $\omega > 1$ 是控制条件强度的引导权重（Weighting Parameter）。

**无缝等价证明**：
仔细观察这个幂次乘积配置，对其两边取对数梯度：
$$ \nabla \log \tilde{q}(\tau_1 | C) = \omega \nabla \log q(\tau_1 | C) + (1-\omega) \nabla \log q(\tau_1) $$
如果我们重新参数化，令 $\omega = 1 + w$，那么 $(1-\omega) = -w$。此时等式就变成了我们在第 3 节中推导的扩散模型 CFG 分数公式的变体，**数学上二者完全严格一致**！这里只是把缩放系数换成了 $\omega$ 的表达形式。

### 6.2 速度场 (Velocity Field) 的修正公式
在扩散模型中，ODE 是沿着“噪声残差 $\epsilon_\theta$”的方向积分演化；而在 Flow Matching 中，马尔可夫演化过程通常直接拟合和预测真实的**速度场 (Velocity Field) $v_t$**。
既然对数密度的线性组合机制相同，那么直接控制转移路线的速度场组合公式也就顺理成章地被推导出来：
$$ \tilde{v}_t(\tau_t, t | C) = (1 - \omega) v_t(\tau_t, t) + \omega v_t(\tau_t, t | C) $$
这个修正后的指导速度 $\tilde{v}_t$ 就是我们最终在推理时，喂给 ODE Solver (求解器) 去积分生成最终轨迹 $\tau_1$ 的导数方向。

### 6.3 训练范式与最优传输 (Optimal Transport) 的结合
在流匹配中获取 $v_t$ 并使得单网络能够执行 CFG，依靠的依然是 **条件丢失 (Condition Dropout)**！

引入一个伯努利分布的掩码变量 $b \in \{0, 1\}$，模型在训练时以一定概率将真实的场景条件 $C$ 替换为全遮蔽的空条件 $H$。为了更具物理意义地优化轨迹，网络可以直接预测终态时间 $t=1$ 时的目标真实轨迹 $\tau_1$：
$$ \mathcal{L}_{flow} = \mathbb{E}_{t \sim U(0,1), b \sim \mathcal{B}, \tau_0, \tau_1} \left\| \tau_\theta\big(\tau_t, t | (1-b) \cdot C + b \cdot H\big) - \tau_1 \right\|^2 $$

> **注：预测轨迹为什么与速度等价？** 
> 按照 Flow Matching 框架中十分流行的**最优传输路径 (Optimal Transport Path)**，假设数据是从纯高斯噪声 $\tau_0$ 到真实目标 $\tau_1$ 走直线的，那么速度就可以简单地通过“末态减初态除以时间差”计算：
> $$ v_\theta(\tau_t, t | C) = \frac{\tau_\theta(\tau_t, t | C) - \tau_0}{t} $$
> (注：在某些公式定义下由 $t$ 从 0 到 1 前向还是反向决定除以 $t$ 还是 $1-t$，核心逻辑均为一次方程直线的斜率项)。利用这个斜率计算就能代回公式 6.2。

### 6.4 业务直觉：为什么行为克隆/自动驾驶需要 CFG？
这一做法解释了 CFG 的一个神妙业务价值，它治愈了直接预测的“过度保守综合征”：
- **纯无条件分布 $v_t(\cdot)$**：模型没输入条件 $C$ (周围邻近车辆信息)，规划时会只按空旷马路直行，容易产生碰撞。
- **常规的有条件分布 $v_t(\cdot | C)$**：模型看到了周围逼近的车辆，模仿学习（BC）天然的特性会导致其产生**反应过度、过于保守 (Overly Conservative)** 的贴边或急刹轨迹。
- **应用 CFG $\tilde{v}_t$**：结合速度公式 $\tilde{v} = v_{uncond} + \omega(v_{cond} - v_{uncond})$。因为 $\omega > 1$，模型实质上在计算 `带条件避让动作 - 无条件直行` 的差值——也就是**纯粹的“避让操作特写净值”**。将该避让特写特征放大加回原底座后，模型不仅学到了需要避让，还能恰到好处、果断且平稳地做出交互决断（Interactive Behavior），大大提升了多模态自动驾驶场景中的表现上限。